## Run DeepEval LLM-as-judge with local LLMs of Ollama

In [ ]:

from deepeval.test_case import LLMTestCase;
from deepeval import evaluate;
from deepeval.metrics import ExactMatchMetric

test_case = LLMTestCase(
    input="What is the capital of France?",
    expected_output="The capital of France is Paris.",
    actual_output="The capital of France is Paris."
)

evaluate([test_case], metrics = [
    ExactMatchMetric()
])

## Test Code LLM as-judge - AnswerRelevancy

In [ ]:
!deepeval set-ollama --model llama3.2:latest

In [ ]:
from dotenv import load_dotenv

from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.evaluate.configs import AsyncConfig
from deepeval.models import OllamaModel
import os


load_dotenv()

ollama_model = OllamaModel( model=os.getenv("LOCAL_OLLAMA_MODEL"), base_url=os.getenv("LOCAL_OLLAMA_BASE_URL"))
    


def test_answer_relevancy():
    test_case = LLMTestCase(
        input="What is the capital of France?",
        expected_output="The capital of France is Paris.",
        actual_output="Paris.",
    )

    metric = AnswerRelevancyMetric(
        model=ollama_model,
        threshold=0.5,
    )

    evaluate(
        test_cases=[test_case],
        metrics=[metric],
        async_config=AsyncConfig(run_async=False),
    )


if __name__ == "__main__":
    test_answer_relevancy()

## Test Code with thinking model in place with OllamaModel Class of DeepEval

In [ ]:
from deepeval.test_case import LLMTestCase;
from deepeval import evaluate;
from deepeval.metrics import AnswerRelevancyMetric
from dotenv import load_dotenv, find_dotenv
from deepeval.evaluate import AsyncConfig
from deepeval.models import OllamaModel
import os

load_dotenv(find_dotenv())

ollama_model = OllamaModel(
    model="llama3.2:latest",
    base_url="http://localhost:11434"
)

test_case = LLMTestCase(
    input="What is the capital of France?",
    expected_output="The capital of France is Paris.",
    actual_output="Chennai."
)

evaluate(test_cases =[test_case], 
         metrics = [AnswerRelevancyMetric(model=ollama_model)],
         async_config=AsyncConfig(run_async=False)
)

## Test Code by Disabling the thinking of model in place with OllamaModel Class of DeepEval

In [ ]:
from deepeval.test_case import LLMTestCase;
from deepeval import evaluate;
from deepeval.metrics import AnswerRelevancyMetric
from dotenv import load_dotenv, find_dotenv
from deepeval.evaluate import AsyncConfig
from deepeval.models import OllamaModel
import os
from typing import Optional, Tuple, Union
from pydantic import BaseModel

load_dotenv(find_dotenv())


class OllamaModelNoThink(OllamaModel):
     def generate(self, prompt: str, schema: Optional[BaseModel] = None) -> Tuple[Union[str, BaseModel], float]:
        chat_model = self.load_model()
        messages = [{"role": "user", "content": prompt}]

        response = chat_model.chat(
            model=self.name,
            messages=messages,
            format=schema.model_json_schema() if schema else None,
            options={
                **{"temperature": self.temperature},
                **self.generation_kwargs,
            },
            think=False
        )
        return (
            (
                schema.model_validate_json(response.message.content)
                if schema
                else response.message.content
            ),
            0,
        )

     async def a_generate(self, prompt: str, schema: Optional[BaseModel] = None) -> Tuple[Union[str, BaseModel], float]:
        chat_model = self.load_model(async_mode=True)
        messages = [{"role": "user", "content": prompt}]

        response = await chat_model.chat(
            model=self.name,
            messages=messages,
            format=schema.model_json_schema() if schema else None,
            options={
                **{"temperature": self.temperature},
                **self.generation_kwargs,
            },
            think=False
        )
        return (
            (
                schema.model_validate_json(response.message.content)
                if schema
                else response.message.content
            ),
            0,
        )

ollama_model = OllamaModelNoThink(
     model="llama3.2:latest",
     base_url="http://localhost:11434"
)

test_case_1 = LLMTestCase(
    input="What is the capital of France?",
    expected_output="The capital of France is Paris.",
    actual_output="Chennai."
)


test_case_2 = LLMTestCase(
    input="Who is the president of the United States?",
    expected_output="The president of the United States is Donald Trump.",
    actual_output="The president of the United States is Joe Biden."
)

evaluate(test_cases =[test_case_1, test_case_2], 
         metrics = [AnswerRelevancyMetric(model=ollama_model)],
         async_config=AsyncConfig(run_async=False)
)

## DataSets

In [ ]:
from deepeval.test_case import LLMTestCase;
from deepeval import evaluate;
from deepeval.metrics import AnswerRelevancyMetric
from dotenv import load_dotenv, find_dotenv
from deepeval.evaluate import AsyncConfig
from deepeval.models import OllamaModel
import os
from typing import Optional, Tuple, Union
from pydantic import BaseModel
from deepeval.dataset import EvaluationDataset

load_dotenv(find_dotenv())


class OllamaModelNoThink(OllamaModel):
     def generate(self, prompt: str, schema: Optional[BaseModel] = None) -> Tuple[Union[str, BaseModel], float]:
        chat_model = self.load_model()
        messages = [{"role": "user", "content": prompt}]

        response = chat_model.chat(
            model=self.name,
            messages=messages,
            format=schema.model_json_schema() if schema else None,
            options={
                **{"temperature": self.temperature},
                **self.generation_kwargs,
            },
            think=False
        )
        return (
            (
                schema.model_validate_json(response.message.content)
                if schema
                else response.message.content
            ),
            0,
        )

     async def a_generate(self, prompt: str, schema: Optional[BaseModel] = None) -> Tuple[Union[str, BaseModel], float]:
        chat_model = self.load_model(async_mode=True)
        messages = [{"role": "user", "content": prompt}]

        response = await chat_model.chat(
            model=self.name,
            messages=messages,
            format=schema.model_json_schema() if schema else None,
            options={
                **{"temperature": self.temperature},
                **self.generation_kwargs,
            },
            think=False
        )
        return (
            (
                schema.model_validate_json(response.message.content)
                if schema
                else response.message.content
            ),
            0,
        )

ollama_model = OllamaModelNoThink(
     model="llama3.2:latest",
     base_url="http://localhost:11434"
)

test_case_1 = LLMTestCase(
    input="What is the capital of France?",
    expected_output="The capital of France is Paris.",
    actual_output="Chennai."
)


test_case_2 = LLMTestCase(
    input="Who is the president of the United States?",
    expected_output="The president of the United States is Donald Trump.",
    actual_output="The president of the United States is Joe Biden."
)

dataset = EvaluationDataset()

dataset.add_test_case(test_case_1)
dataset.add_test_case(test_case_2) 


evaluate(test_cases=dataset.test_cases,
         metrics = [AnswerRelevancyMetric(model=ollama_model)],
         async_config=AsyncConfig(run_async=False)
)

In [ ]:
dataset

## Creating Goldens

In [ ]:
from deepeval.dataset import EvaluationDataset, Golden

golden = Golden(
    input="What is the capital of France?",
    expected_output="The capital of France is Paris.",
    context=[
        "The capital of France is a well-known fact. It is Paris, which is also "
        "the largest city in France and a major cultural and political center."
    ],
)

dataset = EvaluationDataset(goldens=[golden])
# or: dataset = EvaluationDataset(); dataset.add_golden(golden)

In [ ]:
dataset

In [ ]:
test_data = [
    {
        "input": "Who is the current president of the United States of America?",
        "expected_output": "Joe Biden",
    },
    {

        "input": "Who introduced the GPT Model?",
        "expected_output": "Open AI"
    }
]

In [ ]:
goldens = []

for data in test_data:
    golden = Golden(
        input=data["input"],
        expected_output=data["expected_output"],
        context=[
            "The capital of France is a well-known fact. It is Paris, which is also "
            "the largest city in France and a major cultural and political center."
        ],
    )
    goldens.append(golden)
dataset = EvaluationDataset(goldens=goldens)

In [ ]:
dataset

## Creating a Large Adversial Golden DataSet

In [ ]:

import json
from deepeval.dataset import EvaluationDataset, Golden


dataset = EvaluationDataset()

with open("../dev.json", "r") as f:
    data = json.load(f)

for article in data["data"]:
    for para in article["paragraphs"]:
        context = para["context"]
        for qa in para["qas"]:
            expected_output = qa["answers"][0]["text"] if qa.get("answers") else None
            dataset.goldens.append(
                Golden(
                    input=qa["question"],
                    expected_output=expected_output,
                    context=[context],
                )
            )

print(f"Dataset: {len(dataset.goldens)}")

In [ ]:
dataset.goldens[0]

## Convert Golden to Test Case

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage
from deepeval.test_case import LLMTestCase

# This was missing
llm = ChatOllama(model="llama3.2:latest", temperature=0)

dataset.test_cases.clear()

for golden in dataset.goldens[:10]:
    context_text = "\n\n".join(golden.context) if golden.context else ""

    messages = [
        SystemMessage(content=f"Use the following context to answer the question:\n\n{context_text}"),
        HumanMessage(content=golden.input),
    ]

    testcase = LLMTestCase(
        input=golden.input,
        expected_output=golden.expected_output,
        context=golden.context,
        retrieval_context=golden.context,  # needed for Faithfulness / ContextualRelevancy
        actual_output=llm.invoke(messages).content,
    )

    dataset.add_test_case(testcase)

print(f"Test cases: {len(dataset.test_cases)}")
dataset.test_cases

In [ ]:

evaluate(test_cases=dataset.test_cases,
         metrics=[AnswerRelevancyMetric(model=ollama_model)],
         async_config=AsyncConfig(run_async=False)
)